In [11]:
import pandas as pd
import numpy as np

In [27]:
import seaborn as sns
tips = sns.load_dataset("tips")
tips.tail(20)

,total_bill,tip,sex,smoker,day,time,size
224,13.42,1.58,Male,Yes,Fri,Lunch,2
225,16.27,2.50,Female,Yes,Fri,Lunch,2
226,10.09,2.00,Female,Yes,Fri,Lunch,2
227,20.45,3.00,Male,No,Sat,Dinner,4
228,13.28,2.72,Male,No,Sat,Dinner,2
229,22.12,2.88,Female,Yes,Sat,Dinner,2
230,24.01,2.00,Male,Yes,Sat,Dinner,4
231,15.69,3.00,Male,Yes,Sat,Dinner,3
232,11.61,3.39,Male,No,Sat,Dinner,2
233,10.77,1.47,Male,No,Sat,Dinner,2


### #1
- total_bill과 tip 비율(tip_pct = tip / total_bill)을 계산 -> sex * time 조합별로 tip_pct 평균을 비교
- 가장 후하게 팁 주는 조합 찾고 이유 설명



In [61]:
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

#Female * Lunch
fl = tips["tip_pct"][tips["sex"] == "Female"][tips["time"] == "Lunch"].mean().round(3)
#Female * Dinner
fd = tips["tip_pct"][tips["sex"] == "Female"][tips["time"] == "Dinner"].mean().round(3)
#Male * Lunch
ml = tips["tip_pct"][tips["sex"] == "Male"][tips["time"] == "Lunch"].mean().round(3)
#Male * Dinner
md = tips["tip_pct"][tips["sex"] == "Male"][tips["time"] == "Dinner"].mean().round(3)

print(pd.Series([fl, fd, ml, md], index = ["female-lunch", "female-dinner", "male-lunch", "male-dinner"]))
print("\n가장 후한 팁을 주는 부류는 '저녁에 온 여성' 손님")

female-lunch     0.162
female-dinner    0.169
male-lunch       0.166
male-dinner      0.155
dtype: float64

가장 후한 팁을 주는 부류는 '저녁에 온 여성' 손님


### #2
- 파생변수 tip_level 생성:
  - tip_pct(팁 비율)가 상위 20% → "top-tier"
  - tip_pct가 50~80% 분위수 → “mid-tier”
  - 그 외 → “low-tier”
  - 그 후, sex × time × tip_level 조합별 평균 tip_pct 를 계산하고
  - 가장 후하게 팁을 주는 조합 TOP 3를 순서대로 출력
  - 마지막으로, TOP1 조합이 높은 이유를 데이터적 관점에서 해석(ex: total_bill 높음 / size 작음 / dinner vs lunch 차이 등).

In [ ]:
# tips["tip_level"] = pd.cut(lambda x: "top-tier" if tips["tip_pct"].sort_values().head(int(tips["tip_pct"].count()*0.2)) 
#                                else "mid-tier" if tips["tip_pct"].sort_values().tail(int(tips["tip_pct"].count()*0.5) - int(tips["tip_pct"].count()*0.2))
#                                else "low-tier")

tips["tip_level"] = pd.qcut(
    tips["tip_pct"],
    q=[0, 0.2, 0.5, 0.8],
    labels=["top-tier", "mid-tier", "low-tier"])

means = tips.groupby(["sex", "time", "tip_level"])["tip_pct"].mean()
means2 = pd.DataFrame(means)

top3 = means2["tip_pct"].sort_values().head(3)
top3

# top3.head(1)

C:\Users\user\AppData\Local\Temp\ipykernel_3416\3108912483.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = tips.groupby(["sex", "time", "tip_level"])["tip_pct"].mean()


sex     time    tip_level
Female  Dinner  top-tier     0.083671
Male    Dinner  top-tier     0.093749
        Lunch   top-tier     0.095474
Name: tip_pct, dtype: float64

### #3
- 요일(day), 시간대(time), 성별(sex), 흡연(smoker) 조건을 모두 조합해 총 4×2×2×2 = 32개 그룹을 만들고
- 각 그룹의 tip_pct 평균을 계산한 뒤 그룹 내 분산(variance)도 함께 구해라
- 평균 tip_pct가 높으면서 분산은 낮은 (=일관되게 팁을 후하게 주는) "안정적 고팁 그룹(Stably Generous Group)" TOP 3를 선정하라.

In [236]:
groups = tips.groupby(["day", "time", "sex", "smoker"])["tip_pct"].agg(["mean", "var"])

groups.fillna(0)
groups.sort_values(by=["mean", "var"], ascending=[False, True]).head(3)

C:\Users\user\AppData\Local\Temp\ipykernel_3416\3447727633.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groups = tips.groupby(["day", "time", "sex", "smoker"])["tip_pct"].agg(["mean", "var"])


mean       var
day time   sex    smoker                    
Sun Dinner Female Yes     0.237075  0.014459
Fri Dinner Female Yes     0.213179  0.001484
    Lunch  Female Yes     0.203729  0.002814